## Unbiased Dynamic Momentum Strategy
- This implements a momentum long-short strategy. It utilizes the returns of a fixed stock pool over the past N trading days, going long on the best-performing stock and shorting the worst-performing stock at each rebalancing, allocating remaining funds to the cash proxy BIL.

- It uses a custom MomentumLongShort category in AllocationStrategy, reading only previously visible historical data through prices_history to avoid using future information for stock selection.

#### Benchmarks: 
Three buy-and-hold benchmarks are used to examine whether the strategy truly outperforms simple holding.
1. 100% SPY
2. 100% QQQ
3. a balanced portfolio of 70% QQQ + 20% GLD + 10% BIL

## Data Loading and Preparation

In [1]:
import pandas as pd
import numpy as np
from typing import Any
from tiportfolio.allocation import AllocationStrategy
from tiportfolio.allocation import FixRatio
from tiportfolio.calendar import Schedule
from tiportfolio.engine import ScheduleBasedEngine
from tiportfolio.report import compare_strategies

In [ ]:
# Prepare Universe stocks for dynamic stock selection
# UNIVERSE = ['AAPL', 'MSFT', 'NVDA', 'INTC', 'AMD', 'KO', 'PEP', 'PG', 'JNJ', 'WMT', 'META', 'AMZN', 'GOOGL', 
# 'TSLA', 'NFLX', 'JPM', 'C', 'PGR', 'XOM', 'OXY', 'DIS', 'CCL', 'CMCSA', 'HD', 'WHR', 'LEN', 'TGT', 'MCD', 
# 'SBUX', 'WM', 'BA', 'CAT', 'LLY', 'PFE', 'CVX', 'COST']
UNIVERSE = ['AAPL', 'MSFT', 'NVDA', 'INTC', 'AMD', 'META', 'TSLA', 'GOOGL']
CASH = 'BIL'
START = '2018-01-01'
END   = '2024-12-31'

INITIAL_VALUE = 100000
LOOKBACK = 126
TOLERANCE = 0.05
FEE = 0.0035

all_symbols = UNIVERSE + [CASH]

In [ ]:
class MomentumLongShort(AllocationStrategy):
    """
    Each rebalance:
    - Sort by returns over the past lookback_days
    - Go long on the highest-returning stock
    - Go short on the lowest-returning stock
    - Place remaining funds in cash (BIL)
    """
    
    def __init__(self, universe: list[str], cash_symbol: str, lookback_days: int = 126):
        self.universe = universe
        self.cash_symbol = cash_symbol
        self.lookback_days = lookback_days

    def get_symbols(self) -> list[str]:
        return self.universe + [self.cash_symbol]

    def get_target_weights(self, date, total_equity, positions_dollars, prices_row, **context) -> dict[str, float]:
        
        prices_history = context.get("prices_history")
        
        # 1. If no enough data, place in cash and don't trade
        if prices_history is None or len(prices_history) < self.lookback_days + 1:
            return {self.cash_symbol: 1.0}
        
        # 2. Retrieve prices from the past lookback_days days
        hist = prices_history[self.universe].tail(self.lookback_days + 1)
        
        # 3. Calculate the rate of return for this period
        returns = (hist.iloc[-1] / hist.iloc[0]) - 1
        returns = returns.dropna()
        
        # 4. Sort the stocks to find the best and worst performing ones.
        sorted_returns = returns.sort_values(ascending=False)
        long_stock  = sorted_returns.index[0]   # Long: Highest rate of return
        short_stock = sorted_returns.index[-1]  # Short: Lowest rate of return
        
        # 5. 
        return {
            long_stock:        0.5,
            short_stock:      -0.5,
            self.cash_symbol:  1.0,
        }

In [ ]:
# Momentum Strategy
strat_momentum = MomentumLongShort(
    universe=UNIVERSE,
    cash_symbol=CASH,
    lookback_days=LOOKBACK
)
engine_momentum = ScheduleBasedEngine(
    allocation=strat_momentum,
    rebalance=Schedule('month_end'),
    initial_value=INITIAL_VALUE,
    fee_per_share=0.0035
)
res_momentum = engine_momentum.run(symbols=UNIVERSE + [CASH], start=START, end=END)

# Benchmark 1: 100% SPY (Buy & Hold)
strat_spy = FixRatio(weights={'SPY': 1.0})
engine_spy = ScheduleBasedEngine(
    allocation=strat_spy,
    rebalance=Schedule('never'),
    initial_value=INITIAL_VALUE,
    fee_per_share=0
)
res_spy = engine_spy.run(symbols=['SPY'], start=START, end=END)

# Benchmark 2: 100% QQQ (Buy & Hold) 
strat_qqq = FixRatio(weights={'QQQ': 1.0})
engine_qqq = ScheduleBasedEngine(
    allocation=strat_qqq,
    rebalance=Schedule('never'),
    initial_value=INITIAL_VALUE,
    fee_per_share=0
)
res_qqq = engine_qqq.run(symbols=['QQQ'], start=START, end=END)

# Benchmark 3: 70% QQQ + 20% GLD + 10% BIL (Buy & Hold) 
strat_com = FixRatio(weights={'QQQ': 0.7, 'GLD': 0.2, 'BIL': 0.1})
engine_com = ScheduleBasedEngine(
    allocation=strat_com,
    rebalance=Schedule('never'),
    initial_value=INITIAL_VALUE,
    fee_per_share=0
)
res_com = engine_com.run(symbols=['QQQ', 'GLD', 'BIL'], start=START, end=END)

# Comparing 3 results
compare_strategies(res_momentum, res_spy, res_qqq, res_com,
                   names=['Momentum L/S', 'SPY Hold', 'QQQ Hold', 'Balanced QQQ + GLD + BIL'])

Loading bar data...
Loaded bar data: 0:00:03 

Loading bar data...
Loaded bar data: 0:00:01 

Loading bar data...
Loaded bar data: 0:00:01 

Loading bar data...
Loaded bar data: 0:00:02 



,Momentum L/S,SPY Hold,QQQ Hold,Common,better
metric,,,,,
sharpe_ratio,1.019665,0.555964,0.685981,0.691113,Momentum L/S
sortino_ratio,1.552357,0.677882,0.891008,0.902146,Momentum L/S
mar_ratio,0.961856,0.403165,0.549477,0.539212,Momentum L/S
cagr,0.356070,0.136180,0.192355,0.164259,Momentum L/S
max_drawdown,0.370190,0.337778,0.350069,0.304628,Common
